In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
    
from odtlearn.flow_opt import FlowOPT_IPW, FlowOPT_DM, FlowOPT_DR
from odtlearn.datasets import prescriptive_ex_data

# `FlowOPT` Examples

We will look at the different methods of learning optimal prescriptive trees: inverse probability weighting (IPW), direct method (DM), and doubly robust (DR)

## Example 0: Preparing the Data

We will first load a synthetic dataset to use as our running example. In this example, we have precomputed the learned inverse propensity weights (IPW) and predicted outcomes for the direct method (DM). At least one of these values must be computed in order to use any of the prescriptive tree classifiers in ODTLearn. The advantages and disadvantages of using each type of prescriptive tree is given in the subsequent examples, and we refer to Dudik et al. (2011) and Jo et al. (2021) for more details.

The column names in this dataset that correspond to the IPW weights and predicted outcomes for DM are:
- `prob_t_pred_log` is the inverse propensity weight learned through logistic regression
- `prob_t_pred_tree` is the inverse propensity weight learned through a decision tree
- `linear0` is the predicted outcome under treatment 0 learned through linear regression
- `linear1` is the predicted outcome under treatment 1 learned through linear regression
- `lasso0` is the predicted outcome under treatment 0 learned through lasso regression
- `lasso0` is the predicted outcome under treatment 2 learned through lasso regression

In [ ]:
# Read data
train_data, test_data  = prescriptive_ex_data()
print(f'shape{train_data.shape}')
train_data.columns

train_data.head()

**Note**: In practice, users should learn IPW weights and predicted outcomes in the following manner:
- IPW weights can be learned using a standard machine learning method that predicts the probability of treatment assignments given covariates (e.g., logistic regression), then taking its inverse.
- Predicted outcomes for DM are learned by training a machine learning method for each treatment $t$:
    - Take the subset of training data with treatment $t$
    - Learn a machine learning model that can predict the outcome $y$ under treatment $t$ using that subset of training data through, e.g., lasso regression
    - Predict the outcome $y$ under treatment $t$ using the learned model

## Example 1: Inverse Propensity Weighting (IPW)

Inverse Propensity Weighting (IPW) works by creating a pseudo-population where treatment assignments are randomized, allowing us to estimate the average outcome under a learned prescriptive tree.

IPW requires learning propensities, which are the probability that each sample received its assigned treatment. This can be estimated via standard machine learning methods (e.g., logistic regression), where the covariates and assigned treatments are used to create a propensity model that predicts the probability of receiving a treatment.

IPW is only a consistent estimator when the learned propensity model is correctly specified, and may suffer from high variance and sensitivity to small propensity scores (leading to very high inverse propensity weights).

In [ ]:
X = train_data.iloc[:, :20] # covariates
t = train_data["t"] # treatment
y = train_data["y"] # outcome
ipw = train_data["prob_t_pred_tree"] # IPW weights

opt_ipw = FlowOPT_IPW(solver="gurobi", depth=2, time_limit=300)

opt_ipw.fit(X, t, y, ipw) # pass in IPW weights in the fit function

In [ ]:
opt_ipw.print_tree()
fig, ax = plt.subplots(figsize=(4, 4))
opt_ipw.plot_tree(ax=ax, fontsize=10, color_dict={"node": None, "leaves": []})
plt.show()

## Example 2: Direct Method

The Direct Method (DM) works by modeling the outcomes under each treatment directly, which allows for an estimate of the outcome of each data sample under a different treatment assignment.

DM requires learning an outcome model for each treatment in the data. This can be estimated via standard machine learning methods (e.g., lasso regression), where the subset of individuals in the data that received one treatment assignment is used to create the outcome model under that treatment.

DM is only a consistent estimator when the outcome models are correctly specified, and suffers from high bias as it requires extrapolating the outcome beyond the individuals that historically received that treatment.

In [ ]:
X = train_data.iloc[:, :20] # covariates
t = train_data["t"] # treatment
y = train_data["y"] # outcome
y_hat = train_data[["linear0", "linear1"]] # estimated outcomes

opt_dm = FlowOPT_DM(solver="gurobi", depth=2, time_limit=300)

opt_dm.fit(X, t, y, y_hat) # pass in estimated outcomes in the fit function

In [ ]:
opt_dm.print_tree()
fig, ax = plt.subplots(figsize=(4, 4))
opt_dm.plot_tree(ax=ax, fontsize=10, color_dict={"node": None, "leaves": []})
plt.show()

## Example 3: Doubly Robust Method

The Doubly Robust Method (DR) combines the IPW and DM methods. The DR method uses an estimated outcome model (similar to DM), and corrects any bias from the outcome model using IPW. 

DR is consistent if *either* the propensity model OR the outcome model is correctly specified. However, it does require learning both the IPW weights and the outcome models under each treatment.

In [ ]:
X = train_data.iloc[:, :20] # covariates
t = train_data["t"] # treatment
y = train_data["y"] # outcome
ipw = train_data["prob_t_pred_tree"] # IPW weights
y_hat = train_data[["linear0", "linear1"]] # estimated outcomes

opt_dr = FlowOPT_DR(solver="gurobi", depth=2, time_limit=300)

opt_dr.fit(X, t, y, ipw, y_hat) # Pass in both IPW weights and estimated outcomes in the fit function

In [ ]:
opt_dr.print_tree()
fig, ax = plt.subplots(figsize=(4, 4))
opt_dr.plot_tree(ax=ax, fontsize=10, color_dict={"node": None, "leaves": []})
plt.show()

## Example 4: Comparing Performance

It may be of interest to compare the performance of each type of estimator. We can compute summary statistics on the average outcome from each learned tree.

In practice, DR is the typical preferred choice as it is consistent if either IPW or DR methods are consistent. IPW also remains a popular choice since it relies on just one propensity model to learn. We refer to Dudik et al. (2011) and Jo et al. (2021)  for more details.

In [ ]:
# Read test data
print(f'shape{test_data.shape}')
test_data.columns

test_data.head()

In [ ]:
# Predict
X_test = test_data.iloc[:, :20] # covariates
y_0_test = test_data["y0"] # estimated outcome under treatment 0
y_1_test = test_data["y1"] # estimated outcome under treatment 1

predict_ipw = opt_ipw.predict(X_test)
predict_dm = opt_dm.predict(X_test)
predict_dr = opt_dr.predict(X_test)

In [ ]:
# Check the outcome
def calculate_outcome(y0, y1, predict):
    total_outcome = 0
    for i in range(len(predict)):
        if predict[i] == 0:
            total_outcome += y0[i]
        else:
            total_outcome += y1[i]
    return total_outcome / len(predict)

print("IPW test estimated outcome: ", calculate_outcome(y_0_test, y_1_test, predict_ipw))
print("DM test estimated outcome: ", calculate_outcome(y_0_test, y_1_test, predict_dm))
print("DR test estimated outcome: ", calculate_outcome(y_0_test, y_1_test, predict_dr))

## References
* Dudík, M., Langford, J., & Li, L. (2011). Doubly robust policy evaluation and learning. In Proceedings of the 28th International Conference on International Conference on Machine Learning (pp. 1097-1104).
* Jo, N., Aghaei, S., Gómez, A., & Vayanos, P. (2021). Learning optimal prescriptive trees from observational data. arXiv preprint arXiv:2108.13628.